# Notebook 06: TensorFlow TabTransformer

**Objective:** Test whether a transformer over categorical tokens adds value beyond the selected LightGBM baseline and the rejected MLP. Train on dev (2019 to 2021), select on val (2022). Holdout rows are never loaded.

**Decision rule, same as Notebook 05 and fixed before any result here.** Retain the TabTransformer for a target only if all three hold:
1. Its val PR-AUC beats LightGBM in each of 3 training seeds.
2. The paired bootstrap 95% interval for (3-seed ensemble minus LightGBM) excludes zero.
3. The mean PR-AUC gap is at least +0.002.

Otherwise it is recorded as a rejected challenger.

**Budget:** at most 3 settings per target, then 3 seeds of the best one. Same early stopping as the MLP (val PR-AUC, patience 5, best epoch restored), unweighted loss, median imputation fitted on dev, no missingness indicators.

**Design choice:** the data has only 7 categorical columns, so a plain TabTransformer would attend over 7 tokens. The grid therefore also includes a version where the 30 binary flags (comorbidities, medications) are treated as 2-level tokens, giving 37 tokens. This is a design assumption and is tested, not assumed to help.

In [1]:
import sys, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

warnings.filterwarnings("ignore")
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
print("tensorflow", tf.__version__, "| keras", keras.__version__, "| devices:",
      [d.device_type for d in tf.config.list_physical_devices()])

targets = ["adverse_outcome", "readmission_30d"]

# ---- data: holdout rows are filtered out at read time ----
mr = pd.read_parquet("../data/interim/model_ready.parquet", filters=[("split", "!=", "holdout")])
feat_tbl = pd.read_csv("../reports/tables/feature_summary.csv")
features = feat_tbl["feature"].tolist()
cat_features = feat_tbl.loc[feat_tbl["kind"] == "categorical", "feature"].tolist()
binary_features = feat_tbl.loc[feat_tbl["kind"] == "binary", "feature"].tolist()
numeric_features = feat_tbl.loc[feat_tbl["kind"] == "numeric", "feature"].tolist()
assert not {"arrival_day", "arrival_month", "arrival_day_of_week", "arrival_year"} & set(features)

dev = mr[mr["split"] == "dev"].reset_index(drop=True)
val = mr[mr["split"] == "val"].reset_index(drop=True)
del mr

# =============== SHARED FUNCTIONS (copied from Notebooks 04 and 05) ===============
def ece_score(y, p, n_bins=10):
    bins = np.minimum((p * n_bins).astype(int), n_bins - 1)
    return sum((bins == b).mean() * abs(p[bins == b].mean() - y[bins == b].mean())
               for b in range(n_bins) if (bins == b).any())

def paired_boot(y, p_a, p_b, n=200, seed=0):
    rs = np.random.default_rng(seed)
    y, p_a, p_b = np.asarray(y), np.asarray(p_a), np.asarray(p_b)
    diffs = []
    for _ in range(n):
        i = rs.integers(0, len(y), len(y))
        diffs.append(average_precision_score(y[i], p_b[i]) - average_precision_score(y[i], p_a[i]))
    return np.percentile(diffs, [2.5, 50, 97.5]).round(4)

K_BY_TARGET = {"adverse_outcome": [0.05, 0.10, 0.20, 0.30, 0.40],
               "readmission_30d": [0.01, 0.05, 0.10, 0.20]}

def rank_table(y, score, ks, seed=42):
    y, score = np.asarray(y), np.asarray(score)
    jitter = np.random.default_rng(seed).random(len(score)) * 1e-9
    order = np.argsort(-(score + jitter))
    total, prev, rows = int(y.sum()), y.mean(), []
    for k in ks:
        n = int(round(len(y) * k))
        hits = int(y[order[:n]].sum())
        rows.append({"K": k, "n_review": n, "events_captured": hits, "total_events": total,
                     "recall": hits / total, "precision": hits / n,
                     "lift": (hits / n) / prev, "false_positives": n - hits})
    return pd.DataFrame(rows)
# ==================================================================================

# ---- saved val predictions from Notebooks 04 and 05 ----
vp = pd.read_parquet("../data/interim/val_predictions_classical.parquet")
mp = pd.read_parquet("../data/interim/val_predictions_mlp.parquet")
assert (vp["encounter_id"].values == val["encounter_id"].values).all(), "classical row order differs"
assert (mp["encounter_id"].values == val["encounter_id"].values).all(), "mlp row order differs"
for t in targets:
    print(t, "| LightGBM:", round(average_precision_score(val[t], vp[f"lightgbm_{t}"]), 4),
          "| MLP ensemble:", round(average_precision_score(val[t], mp[f"mlp_{t}"]), 4))

print()
print("features:", len(features), "| numeric", len(numeric_features), "| binary", len(binary_features),
      "| categorical", len(cat_features))
print("dev:", dev.shape, "years", sorted(int(y) for y in dev["arrival_year"].unique()))
print("val:", val.shape, "years", sorted(int(y) for y in val["arrival_year"].unique()))
print("splits present:", sorted(set(dev["split"]) | set(val["split"])))
print("categories per categorical column (dev):", {c: int(dev[c].nunique()) for c in cat_features})

tensorflow 2.20.0 | keras 3.11.3 | devices: ['CPU']
adverse_outcome | LightGBM: 0.7553 | MLP ensemble: 0.7501
readmission_30d | LightGBM: 0.2138 | MLP ensemble: 0.2123

features: 90 | numeric 53 | binary 30 | categorical 7
dev: (400365, 105) years [2019, 2020, 2021]
val: (133853, 105) years [2022]
splits present: ['dev', 'val']
categories per categorical column (dev): {'city': 25, 'urban_rural': 3, 'hospital_type': 5, 'gender': 2, 'insurance_type': 5, 'arrival_mode': 7, 'chief_complaint': 30}


In [2]:
# ---- preprocessing, fitted on dev only (same as Notebook 05) ----
num_med  = dev[numeric_features].median()
num_fill = dev[numeric_features].fillna(num_med)
num_mean = num_fill.mean()
num_std  = num_fill.std().replace(0, 1.0)

cat_maps  = {c: {v: i + 1 for i, v in enumerate(sorted(dev[c].unique()))} for c in cat_features}
cat_sizes = [len(cat_maps[c]) + 1 for c in cat_features]          # +1: code 0 = unseen category
cat_off   = np.concatenate([[0], np.cumsum(cat_sizes)[:-1]]).astype("int32")
bin_off   = (sum(cat_sizes) + 2 * np.arange(len(binary_features))).astype("int32")
vocab_total = sum(cat_sizes) + 2 * len(binary_features)

assert set(np.unique(dev[binary_features].values)) <= {0, 1}
assert set(np.unique(val[binary_features].values)) <= {0, 1}

def make_inputs(d, bin_tokens):
    Xn = ((d[numeric_features].fillna(num_med) - num_mean) / num_std).astype("float32").values
    codes = np.column_stack([d[c].map(cat_maps[c]).fillna(0).astype("int32").values for c in cat_features])
    tok = codes + cat_off
    Xb = d[binary_features].astype("float32").values
    if bin_tokens:
        tok = np.hstack([tok, Xb.astype("int32") + bin_off])
        dense = Xn
    else:
        dense = np.hstack([Xn, Xb])
    return {"tokens": tok.astype("int32"), "num": dense.astype("float32")}

inp = {b: (make_inputs(dev, b), make_inputs(val, b)) for b in (False, True)}
for b, (xd, xv) in inp.items():
    assert not np.isnan(xd["num"]).any() and not np.isnan(xv["num"]).any()
    assert xd["tokens"].max() < vocab_total and xv["tokens"].max() < vocab_total
    print("binary flags as tokens" if b else "categoricals only", "| tokens:", xd["tokens"].shape[1],
          "| dense inputs:", xd["num"].shape[1])
print("vocabulary size:", vocab_total,
      "| unseen category codes in val:", int((inp[False][1]["tokens"][:, :7] == cat_off).sum()))

# ---- model ----
def build_tt(bin_tokens, d_model=32, heads=4, n_layers=2, ff_mult=2, attn_dropout=0.1,
             mlp=(128, 64), mlp_dropout=0.2, lr=1e-3):
    n_tok = len(cat_features) + (len(binary_features) if bin_tokens else 0)
    n_dense = len(numeric_features) + (0 if bin_tokens else len(binary_features))
    tok_in = keras.Input(shape=(n_tok,), dtype="int32", name="tokens")
    num_in = keras.Input(shape=(n_dense,), name="num")
    x = layers.Embedding(vocab_total, d_model)(tok_in)
    for _ in range(n_layers):
        a = layers.MultiHeadAttention(num_heads=heads, key_dim=d_model // heads, dropout=attn_dropout)(x, x)
        x = layers.LayerNormalization(epsilon=1e-6)(layers.Add()([x, a]))
        f = layers.Dense(d_model * ff_mult, activation="gelu")(x)
        f = layers.Dropout(attn_dropout)(layers.Dense(d_model)(f))
        x = layers.LayerNormalization(epsilon=1e-6)(layers.Add()([x, f]))
    h = layers.Concatenate()([layers.Flatten()(x), layers.LayerNormalization(epsilon=1e-6)(num_in)])
    for u in mlp:
        h = layers.Dense(u)(h)
        h = layers.BatchNormalization()(h)
        h = layers.Activation("relu")(h)
        h = layers.Dropout(mlp_dropout)(h)
    out = layers.Dense(1, activation="sigmoid")(h)
    m = keras.Model([tok_in, num_in], out)
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(curve="PR", name="auc_pr")])
    return m

SETTINGS = {1: dict(bin_tokens=False, d_model=32, heads=4, n_layers=2),
            2: dict(bin_tokens=True,  d_model=32, heads=4, n_layers=2),
            3: dict(bin_tokens=True,  d_model=16, heads=4, n_layers=1)}

# ---- timing and sanity run: setting 2, 2 epochs, adverse_outcome only ----
t = "adverse_outcome"
xd, xv = inp[SETTINGS[2]["bin_tokens"]]
keras.utils.set_random_seed(0)
model = build_tt(**SETTINGS[2])
print("parameters:", model.count_params())

t0 = time.time()
model.fit(xd, dev[t].values.astype("float32"),
          validation_data=(xv, val[t].values.astype("float32")),
          epochs=2, batch_size=1024, verbose=2)
print("seconds for 2 epochs:", round(time.time() - t0))

p = model.predict(xv, batch_size=4096, verbose=0).ravel()
print("val PR-AUC (sklearn):", round(average_precision_score(val[t], p), 4),
      "| ROC-AUC:", round(roc_auc_score(val[t], p), 4), "| prevalence:", round(val[t].mean(), 4))

categoricals only | tokens: 7 | dense inputs: 83
binary flags as tokens | tokens: 37 | dense inputs: 53
vocabulary size: 144 | unseen category codes in val: 0
parameters: 189355
Epoch 1/2
391/391 - 99s - 252ms/step - auc_pr: 0.7034 - loss: 0.5660 - val_auc_pr: 0.7369 - val_loss: 0.5451
Epoch 2/2
391/391 - 88s - 225ms/step - auc_pr: 0.7331 - loss: 0.5397 - val_auc_pr: 0.7402 - val_loss: 0.5352
seconds for 2 epochs: 187
val PR-AUC (sklearn): 0.7402 | ROC-AUC: 0.7952 | prevalence: 0.408


In [3]:
tt_preds, tt_log = {}, []
lgb_ref = {t: average_precision_score(val[t], vp[f"lightgbm_{t}"].values) for t in targets}

def run_tt(setting, t, seed):
    xd, xv = inp[SETTINGS[setting]["bin_tokens"]]
    keras.utils.set_random_seed(seed)
    m = build_tt(**SETTINGS[setting])
    cb = keras.callbacks.EarlyStopping(monitor="val_auc_pr", mode="max", patience=5,
                                       restore_best_weights=True)
    t0 = time.time()
    h = m.fit(xd, dev[t].values.astype("float32"),
              validation_data=(xv, val[t].values.astype("float32")),
              epochs=40, batch_size=1024, callbacks=[cb], verbose=0)
    p = m.predict(xv, batch_size=4096, verbose=0).ravel()
    y = val[t].values
    tt_preds[(setting, t, seed)] = p
    tt_log.append({"setting": setting, "target": t, "seed": seed,
                   "best_epoch": int(np.argmax(h.history["val_auc_pr"])) + 1,
                   "epochs_run": len(h.history["loss"]),
                   "pr_auc": average_precision_score(y, p), "roc_auc": roc_auc_score(y, p),
                   "brier": brier_score_loss(y, p), "ece": ece_score(y, p),
                   "secs": round(time.time() - t0)})
    print("done: setting", setting, t, "| seconds:", tt_log[-1]["secs"], "| epochs:", tt_log[-1]["epochs_run"])

for s in [1, 3]:
    for t in targets:
        run_tt(s, t, 0)

log = pd.DataFrame(tt_log)
log["lightgbm_pr_auc"] = log["target"].map(lgb_ref)
log["gap_vs_lightgbm"] = log["pr_auc"] - log["lightgbm_pr_auc"]
print()
print(log.round(4).to_string(index=False))

done: setting 1 adverse_outcome | seconds: 487 | epochs: 14
done: setting 1 readmission_30d | seconds: 299 | epochs: 10
done: setting 3 adverse_outcome | seconds: 473 | epochs: 15
done: setting 3 readmission_30d | seconds: 763 | epochs: 11

 setting          target  seed  best_epoch  epochs_run  pr_auc  roc_auc  brier    ece  secs  lightgbm_pr_auc  gap_vs_lightgbm
       1 adverse_outcome     0           9          14  0.7455   0.7977 0.1769 0.0068   487           0.7553          -0.0098
       1 readmission_30d     0           5          10  0.2104   0.6022 0.1151 0.0184   299           0.2138          -0.0034
       3 adverse_outcome     0          10          15  0.7433   0.7968 0.1774 0.0050   473           0.7553          -0.0120
       3 readmission_30d     0           6          11  0.2072   0.6006 0.1149 0.0105   763           0.2138          -0.0067


In [4]:
run_tt(2, "adverse_outcome", 0)

log = pd.DataFrame(tt_log)
log["lightgbm_pr_auc"] = log["target"].map(lgb_ref)
log["gap_vs_lightgbm"] = log["pr_auc"] - log["lightgbm_pr_auc"]
print()
print(log.sort_values(["target", "setting"]).round(4).to_string(index=False))

best_setting = {t: int(log[log["target"] == t].sort_values("pr_auc").iloc[-1]["setting"]) for t in targets}
print()
print("best setting by val PR-AUC:", best_setting)

done: setting 2 adverse_outcome | seconds: 1484 | epochs: 13

 setting          target  seed  best_epoch  epochs_run  pr_auc  roc_auc  brier    ece  secs  lightgbm_pr_auc  gap_vs_lightgbm
       1 adverse_outcome     0           9          14  0.7455   0.7977 0.1769 0.0068   487           0.7553          -0.0098
       2 adverse_outcome     0           8          13  0.7433   0.7968 0.1776 0.0161  1484           0.7553          -0.0120
       3 adverse_outcome     0          10          15  0.7433   0.7968 0.1774 0.0050   473           0.7553          -0.0120
       1 readmission_30d     0           5          10  0.2104   0.6022 0.1151 0.0184   299           0.2138          -0.0034
       3 readmission_30d     0           6          11  0.2072   0.6006 0.1149 0.0105   763           0.2138          -0.0067

best setting by val PR-AUC: {'adverse_outcome': 1, 'readmission_30d': 1}


In [5]:
lgb_p = {t: vp[f"lightgbm_{t}"].values for t in targets}
lr_p  = {t: vp[f"logreg_{t}"].values for t in targets}
mlp_p = {t: mp[f"mlp_{t}"].values for t in targets}

print("Rule 1 uses seed 0 only (shortcut): seeds 1 and 2 are run only if seed 0 beats LightGBM.\n")
for t in targets:
    s = best_setting[t]
    p = tt_preds[(s, t, 0)]
    pr = average_precision_score(val[t], p)
    gap = pr - lgb_ref[t]
    c1 = bool(pr > lgb_ref[t])
    ci = paired_boot(val[t], lgb_p[t], p)
    c2 = bool(ci[0] > 0)
    c3 = bool(gap >= 0.002)
    print(t, "| setting", s, "| seed 0 PR-AUC:", round(pr, 4), "| LightGBM:", round(lgb_ref[t], 4))
    print("  gap vs LightGBM:", round(gap, 4))
    print("  TabTransformer minus LightGBM [2.5%, median, 97.5%]:", ci)
    print("  TabTransformer minus LogReg   [2.5%, median, 97.5%]:", paired_boot(val[t], lr_p[t], p))
    print("  TabTransformer minus MLP-ens  [2.5%, median, 97.5%]:", paired_boot(val[t], mlp_p[t], p))
    print("  rule 1:", c1, "| rule 2:", c2, "| rule 3:", c3, "->",
          "RETAIN (run seeds 1 and 2 before final verdict)" if (c1 and c2 and c3) else "REJECT")
    print()

Rule 1 uses seed 0 only (shortcut): seeds 1 and 2 are run only if seed 0 beats LightGBM.

adverse_outcome | setting 1 | seed 0 PR-AUC: 0.7455 | LightGBM: 0.7553
  gap vs LightGBM: -0.0098
  TabTransformer minus LightGBM [2.5%, median, 97.5%]: [-0.0109 -0.0098 -0.0088]
  TabTransformer minus LogReg   [2.5%, median, 97.5%]: [-0.0019 -0.001  -0.0001]
  TabTransformer minus MLP-ens  [2.5%, median, 97.5%]: [-0.0054 -0.0047 -0.004 ]
  rule 1: False | rule 2: False | rule 3: False -> REJECT

readmission_30d | setting 1 | seed 0 PR-AUC: 0.2104 | LightGBM: 0.2138
  gap vs LightGBM: -0.0034
  TabTransformer minus LightGBM [2.5%, median, 97.5%]: [-0.0053 -0.0035 -0.0015]
  TabTransformer minus LogReg   [2.5%, median, 97.5%]: [-0.0064 -0.0044 -0.0024]
  TabTransformer minus MLP-ens  [2.5%, median, 97.5%]: [-0.0038 -0.0019  0.0001]
  rule 1: False | rule 2: False | rule 3: False -> REJECT



In [6]:
tables = Path("../reports/tables"); tables.mkdir(parents=True, exist_ok=True)

log = pd.DataFrame(tt_log)
log["lightgbm_pr_auc"] = log["target"].map(lgb_ref)
log["gap_vs_lightgbm"] = log["pr_auc"] - log["lightgbm_pr_auc"]
log["architecture"] = log["setting"].map({1: "7 tokens, d=32, 2 layers",
                                          2: "37 tokens, d=32, 2 layers",
                                          3: "37 tokens, d=16, 1 layer"})
log.round(4).to_csv(tables / "tabtransformer_runs.csv", index=False)

print(log[["setting", "target", "seed", "pr_auc", "gap_vs_lightgbm", "secs"]].round(4).to_string(index=False))
print("rows:", len(log), "| seeds:", sorted(log["seed"].unique()))
print("splits in memory:", sorted(set(dev["split"]) | set(val["split"])))
p = tables / "tabtransformer_runs.csv"
print("tabtransformer_runs.csv | KB:", round(p.stat().st_size / 1e3, 1) if p.exists() else "MISSING")

 setting          target  seed  pr_auc  gap_vs_lightgbm  secs
       1 adverse_outcome     0  0.7455          -0.0098   487
       1 readmission_30d     0  0.2104          -0.0034   299
       3 adverse_outcome     0  0.7433          -0.0120   473
       3 readmission_30d     0  0.2072          -0.0067   763
       2 adverse_outcome     0  0.7433          -0.0120  1484
rows: 5 | seeds: [np.int64(0)]
splits in memory: ['dev', 'val']
tabtransformer_runs.csv | KB: 0.6


## Setup
Train on dev (2019 to 2021, 400,365 rows), evaluate on val (2022, 133,853 rows). Holdout rows were filtered out at read time. 90 features: 53 numeric (median fill and standardisation fitted on dev), 7 categorical, 30 binary. Tokens index one shared embedding table of 144 entries (one block per column, code 0 reserved for unseen categories, none occur in val). Self-attention blocks (4 heads, residual connections, layer norm, feed-forward) over the tokens, numeric features layer-normalised and concatenated with the flattened token output, then dense layers (128, 64) with batch normalisation, ReLU and dropout 0.2. Unweighted binary cross-entropy, Adam at 1e-3, batch size 1024, up to 40 epochs, early stopping on val PR-AUC with patience 5 and best-epoch weights restored.

## Decision rule (written before any result)
Retain the TabTransformer for a target only if (1) val PR-AUC beats LightGBM in each of 3 seeds, (2) the paired bootstrap 95% interval for the 3-seed ensemble minus LightGBM excludes zero, (3) the mean PR-AUC gap is at least +0.002. Budget: 3 settings per target, then 3 seeds of the best one.

Shortcut added before any run (it follows from rule 1): if seed 0 of the best setting does not beat LightGBM, rule 1 fails regardless of other seeds, so seeds 1 and 2 are skipped.

## Settings tried (seed 0, reports/tables/tabtransformer_runs.csv)
| Setting | Tokens | d_model | Layers | adverse_outcome PR-AUC | readmission_30d PR-AUC |
|---|---:|---:|---:|---:|---:|
| 1 | 7 | 32 | 2 | 0.7455 | 0.2104 |
| 2 | 37 | 32 | 2 | 0.7433 | not run |
| 3 | 37 | 16 | 1 | 0.7433 | 0.2072 |

LightGBM: 0.7553 and 0.2138. Run times: setting 1 took 487 s and 299 s, setting 2 took 1,484 s, setting 3 took 473 s and 763 s.

Deviation from the grid: setting 2 was not run for readmission_30d. That decision was made after settings 1 and 3 had been seen, and before setting 2 ran. Both were already 0.0034 or more below LightGBM, and setting 2 on adverse_outcome cost about 25 minutes. It was skipped for time and no result is behind it.

## Result: TabTransformer rejected for both targets
- adverse_outcome (setting 1): 0.7455. Minus LightGBM: -0.0098 (95% interval -0.0109 to -0.0088). Minus Logistic Regression: median -0.0010 (-0.0019 to -0.0001). Minus MLP 3-seed ensemble: median -0.0047 (-0.0054 to -0.0040).
- readmission_30d (setting 1): 0.2104. Minus LightGBM: -0.0034 (-0.0053 to -0.0015). Minus Logistic Regression: median -0.0044 (-0.0064 to -0.0024). Minus MLP ensemble: median -0.0019 (-0.0038 to +0.0001).
- Seed 0 of the best setting is below LightGBM for both targets, so all three rules fail. Seeds 1 and 2 were skipped under the shortcut.
- Binary flags as tokens did not help: setting 2 scored 0.7433 against 0.7455 for setting 1 (same size otherwise) and took 1,484 s against 487 s. No interval was computed for that single-seed gap.
- The TabTransformer is recorded as a rejected challenger. LightGBM stays the selected model for both targets.

## Model ladder, validation PR-AUC
| Model | adverse_outcome | readmission_30d |
|---|---:|---:|
| Constant prevalence | 0.4080 | 0.1360 |
| Logistic Regression (age spline) | 0.7463 | 0.2148 |
| XGBoost | 0.7548 | 0.2138 |
| LightGBM | 0.7553 | 0.2138 |
| MLP (mean of 3 seeds) | 0.7494 | 0.2106 |
| TabTransformer (1 seed) | 0.7455 | 0.2104 |

## Limitations
- One seed per setting. Seed spread was 0.0000 to 0.0008 for the MLP, but it was not measured for this architecture.
- Three settings of one architecture family, setting 2 for adverse_outcome only. This does not show that no transformer could match LightGBM.
- The comparison with the MLP ensemble favours the MLP (3-seed ensemble against one seed).
- Intervals measure sampling noise on val 2022 only.
- Recall@K and Precision@K tables were produced only for the classical models. The two rejected deep models are compared on PR-AUC, ROC-AUC, Brier and ECE.

## Moves to Notebook 07
- Model objects were not saved. Refit LightGBM on dev with the same parameters and 291 trees (adverse_outcome) and 51 trees (readmission_30d), and confirm val PR-AUC reproduces 0.7553 and 0.2138.
- Calibration design, settled before any result: a calibrator fitted on val and scored on the same rows is optimistic. Split val chronologically by arrival_ts, fit on the first half, score on the second, and compare uncalibrated, Platt and isotonic.
- Recompute Recall@K and Precision@K after the chosen calibration. Isotonic can create tied scores, so keep the random tie-break.
- SHAP on the LightGBM model for both targets, with local examples (true positive, false positive, false negative, low risk). Contributions describe model behaviour, not causes.